## Script for Comparing Incorrectly Labeled Epochs

In [98]:
import mne 
import os
import numpy as np 
import pandas as pd
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import pickle
from sklearn.model_selection import train_test_split, KFold
import matplotlib
matplotlib.use('QtAgg') 

In [99]:
# importing model 
with open("M2_all.pkl", "rb") as f:
    model = pickle.load(f)

KeyboardInterrupt: 

In [ ]:
# importing features dataframe 
features_all = pd.read_pickle("training_features_19032026.pkl")

In [ ]:
X_zygo = features_all[["Zygo"]] 
X_corr = features_all[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

y_zygo = features_all["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all["Num_Contractions_Corr"].astype(int).to_numpy()

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate((y_zygo,y_corr),axis=0)       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

In [ ]:
n1 = len(X_zygo)
labels = np.where(idx_test < n1, "Zygo", "Corr")

idx_test_1 = idx_test[idx_test < n1]
idx_test_2 = idx_test[idx_test >= n1] - n1
X_test_zygo = X[idx_test_1]
X_test_corr = X[idx_test_2]
X_test = X[idx_test]

# get testing results 
y_pred_test_zygo = model.predict(X_test_zygo)   
y_pred_test_corr = model.predict(X_test_corr)   
y_pred_test  = model.predict(X_test)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)   
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)   
y_pred_test = np.argmax(y_pred_test,axis=1)

# get other fields for dataframe 
subjects = features_all["Subject"] 
subject_subset = np.concatenate([subjects[idx_test_1],subjects[idx_test_2]])

epochs = features_all["Triggers_Order_Nap"] 
epochs_subset = np.concatenate([epochs[idx_test_1],epochs[idx_test_2]])

naps = features_all["Nap Number"] 
naps_subset = np.concatenate([naps[idx_test_1],naps[idx_test_2]])

# get mismatched epoch indices where the tested indices dont equal predicted 
epoch_idx = np.where(y[idx_test] != y_pred_test)

mismatched_subj = subject_subset[epoch_idx]
mismatched_epochs = epochs_subset[epoch_idx]
mismatched_naps = naps_subset[epoch_idx]

Pre-Processing

In [ ]:
# making dataframe for epoch rescoring 
rescore_epochs = pd.DataFrame({
    "Subject": mismatched_subj,
    "Nap Number": mismatched_naps,
    "Triggers_Order_Nap":mismatched_epochs,
    "Label": labels[epoch_idx],
    "Prediction": y_pred_test[epoch_idx]
}, index=epoch_idx[0])

rescore_epochs_reshape = (
    rescore_epochs
    .pivot_table(
        index=["Subject", "Nap Number", "Triggers_Order_Nap"],
        columns="Label",
        values="Prediction",
        aggfunc="first"
    )
    .rename(columns={
        "Zygo": "Prediction_Zygo",
        "Corr": "Prediction_Corr"
    })
    .reset_index()
)

In [ ]:
# making final dataframe for going through bad epochs 
rows = []
seen = set()

for _, row in rescore_epochs_reshape.iterrows():
    key = (row["Subject"], row["Nap Number"], row["Triggers_Order_Nap"])

    # skip duplicates
    if key in seen:
        continue
    seen.add(key)

    # find matching row(s) in the other dataframe
    match = features_all[
        (features_all["Subject"] == row["Subject"]) &
        (features_all["Nap Number"] == row["Nap Number"]) &
        (features_all["Triggers_Order_Nap"] == row["Triggers_Order_Nap"])
    ]

    # if there is a match, take the first one
    if not match.empty:
        match_row = match.iloc[0]

        # --- HANDLE NaNs HERE ---
        match_zygo = match_row["Zygo"]
        match_corr = match_row["Corr"]

        pred_zygo = row["Prediction_Zygo"]
        pred_corr = row["Prediction_Corr"]

        if pd.isna(pred_zygo):
            pred_zygo = np.argmax(model.predict(np.array(match_zygo.tolist()).reshape(1, 2251,1), verbose=0))

        if pd.isna(pred_corr):
            pred_corr = np.argmax(model.predict(np.array(match_corr.tolist()).reshape(1, 2251,1), verbose=0))


        rows.append({
            "Subject": row["Subject"],
            "Nap Number": row["Nap Number"],
            "Triggers_Order_Nap": row["Triggers_Order_Nap"],
            "Prediction_Zygo": pred_zygo,
            "Prediction_Corr": pred_corr,

            # examples of fields from df_other
            "Num_Contractions_Zygo": match_row["Num_Contractions_Zygo"],
            "Num_Contractions_Corr": match_row["Num_Contractions_Corr"],
            "Zygo": match_row["Zygo"],
            "Corr": match_row["Corr"],
        })

    else:
        # if no match exists, still keep the row
        rows.append({
            "Subject": row["Subject"],
            "Nap Number": row["Nap Number"],
            "Triggers_Order_Nap": row["Triggers_Order_Nap"],
            "Label": row["Label"],
            "Prediction_Zygo": pred_zygo,
            "Prediction_Corr": pred_corr,

            "Num_Contractions_Zygo": pd.NA,
            "Num_Contractions_Corr": pd.NA,
            "Zygo": pd.NA,
            "Corr": pd.NA
        })


mismatch_df = pd.DataFrame(rows)    


In [ ]:
mismatch_df.head(10)


Scoring for Mismatched Epochs

In [148]:
#GUI
frq = 250 
current_index=0
inter_trigger_length=10
window = 50
step = 1

# defining triggers 
df_triggers = pd.DataFrame(
    index=mismatch_df.index,
    columns=["Epoch",
        "Contraction_number",
        "Response_start_sample",
        "Response_end_sample",
        "Muscle_type",
        "RT_sec",
    ]
)

def plot_figure(t):
    global frq 
    global mismatch_df,df_triggers
 
    fig, ax = plt.subplots(2, 1) #, figsize=(10, 5))

    zygo = mismatch_df["Zygo"].iloc[t] 
    corr = mismatch_df["Corr"].iloc[t] 

    fig.suptitle(f"Epoch {t + 1}")
    if df_triggers.loc[t, 'Contraction_number'] != None and df_triggers.loc[t, 'Response_end_sample'] != None and df_triggers.loc[t, 'Response_start_sample'] != None:
        fdct = {'color': 'r'}
        title = fig.suptitle(f"Epoch {t + 1} - SCORED : {df_triggers.loc[t, 'Contraction_number']} CONTRACTIONS; START = {df_triggers.loc[t, 'Response_start_sample']}; END = {df_triggers.loc[t, 'Response_end_sample']}")
        title.set(**fdct)
    elif df_triggers.loc[t, 'Contraction_number'] ==0 and df_triggers.loc[t, 'Response_end_sample'] == None and df_triggers.loc[t, 'Response_start_sample'] == None:
        fdct = {'color': 'r'}
        title = fig.suptitle(f"Epoch {t + 1} - SCORED : {df_triggers.loc[t, 'Contraction_number']} CONTRACTIONS")
        title.set(**fdct)
    else:
        title = fig.suptitle(f"Epoch {t + 1} - UNFINISHED SCORING : {df_triggers.loc[t, 'Contraction_number']} CONTRACTIONS; START = {df_triggers.loc[t, 'Response_start_sample']}; END = {df_triggers.loc[t, 'Response_end_sample']}")


    # Corr subplot
    ax[0].plot(corr, color="blue", label="Corr")

    #ax[0].set_ylim(-ymax_emg, ymax_emg)

    ax[0].set_ylabel("Corr EMG [V]")
    ax[0].set_xlabel("Samples")

    # Zygo subplot
    ax[1].plot(zygo, label="Zygo",color="black")
    #ax[1].set_ylim(-ymax_emg, ymax_emg)

    ax[1].set_ylabel("Zygo EMG [V]")
    ax[1].set_xlabel("Samples")


    # Connect mouse click and key press events
    fig.canvas.mpl_connect('key_press_event', on_key)
    fig.canvas.mpl_connect('button_press_event', on_click)

    plt.tight_layout()
    plt.show()


def on_click(event):
    global current_index, fig, df_triggers,i
    if event.inaxes:  # Check if click occurred in any axes

        for i, a in enumerate(event.canvas.figure.axes):
            if event.inaxes == a:
                clicked_muscle = "Corr" if i == 0 else "Zygo"
                break
          
        x = int(event.xdata)

        # identify metadata 
        epoch = current_index
        subject = mismatch_df.loc[current_index, "Subject"]
        nap = mismatch_df.loc[current_index, "Nap Number"]
        trigger = mismatch_df.loc[current_index, "Triggers_Order_Nap"]
        stim = mismatch_df.loc[current_index, "Triggers_Order_Nap"]  # or your real stim sample field

        mask = ((df_triggers["Epoch"] == current_index) &
            (df_triggers["Muscle_type"] == clicked_muscle))

        if not mask.any():
            print(clicked_muscle, x)
            new_row = {
                "Epoch": current_index,
                "Contraction_number": np.nan,
                "Response_start_sample": x,
                "Response_end_sample": np.nan,
                "Stim_time_sample": stim,
                "Muscle_type": clicked_muscle,
                "RT_sec": (x - stim) / frq,
            }
            df_triggers.loc[current_index+i] = new_row

        else:
            row_idx = df_triggers.index[mask][0]

            if pd.isna(df_triggers.loc[row_idx, "Response_start_sample"]):
                df_triggers.loc[row_idx, "Response_start_sample"] = x
                df_triggers.loc[row_idx, "RT_sec"] = (
                    x - df_triggers.loc[row_idx, "Stim_time_sample"]
                ) / frq
            else:
                df_triggers.loc[row_idx, "Response_end_sample"] = x

        plt.close()  # Close current figure
        plot_figure(current_index)         
             

# Keyboard press event handler
def on_key(event):
    global current_index, fig, mismatch_df,df_triggers,i
    allowed_keys = {'0','1', '2', '3', '4', '5','5', '7', '8', '9'}
    key = event.key
        
    if key in allowed_keys:
        df_triggers.loc[current_index, 'Contraction_number'] = int(key) 
        df_triggers.loc[current_index+i, 'Contraction_number'] = int(key) 
        if (df_triggers.loc[current_index, 'Response_end_sample'] != None and df_triggers.loc[current_index, 'Response_start_sample'] != None) or (df_triggers.loc[current_index, 'Contraction_number'] ==0):
            plt.close()  # Close current figure
            plot_figure(current_index) 
    elif event.key == 'right':  # Move to next figure
        current_index = (current_index + 1) % len(mismatch_df)  # Loop to the start
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure
    elif event.key == 'left':  # Move to previous figure
        current_index = (current_index - 1) % len(mismatch_df)  # Loop to the end
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure
    elif event.key == 'r':  # Move to previous figure
        df_triggers.loc[current_index, 'Response_start_sample'] = None
        df_triggers.loc[current_index, 'Response_end_sample'] = None 
        df_triggers.loc[current_index, 'Contraction_number'] = None
        df_triggers.loc[current_index, 'Muscle_type'] = None
        df_triggers.loc[current_index, 'RT_sec'] = None

        df_triggers.loc[current_index+i, 'Response_start_sample'] = None
        df_triggers.loc[current_index+i, 'Response_end_sample'] = None 
        df_triggers.loc[current_index+i, 'Contraction_number'] = None
        df_triggers.loc[current_index+i, 'Muscle_type'] = None
        df_triggers.loc[current_index+i, 'RT_sec'] = None
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'q':  # Custom action for specific key
        print("Quitting the plot!")
        plt.close()

    elif event.key == 'escape':  
        print("Quitting the plot!")
        plt.close(fig)  
        

# Plot the first figure
plot_figure(current_index)

Corr 251


QCoreApplication::exec: The event loop is already running
QCoreApplication::exec: The event loop is already running
QCoreApplication::exec: The event loop is already running


Zygo 86


QCoreApplication::exec: The event loop is already running
QCoreApplication::exec: The event loop is already running
QCoreApplication::exec: The event loop is already running
QCoreApplication::exec: The event loop is already running


Corr 392


...

Zygo 639


QCoreApplication::exec: The event loop is already running
QCoreApplication::exec: The event loop is already running
QCoreApplication::exec: The event loop is already running
QCoreApplication::exec: The event loop is already running


Corr 502


QCoreApplication::exec: The event loop is already running


KeyboardInterrupt: 

In [149]:
df_triggers.head()

,Epoch,Contraction_number,Response_start_sample,Response_end_sample,Muscle_type,RT_sec
0,0,2,251,734,Corr,0.876
1,1,4,392,1720,Corr,1.384
2,2,NaN,502,NaN,Corr,2.004
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN
